# KlippEye — il modello locale del monitor IA

Questo notebook addestra il piccolo modello che girerà sul Raspberry. Esegui le celle **una alla volta, dall'alto in basso**
(il tasto ▶ a sinistra di ogni cella). Prima di tutto: menu **Runtime → Cambia tipo di runtime → GPU (T4)** → Salva.

Ti servono:
1. una **chiave Roboflow** gratuita (per scaricare le raccolte pubbliche): [roboflow.com](https://roboflow.com) → registrati →
   *Settings* → *API Keys* → copia la **Private API Key**;
2. facoltativo ma consigliato: la **cartella dei dati di KlippEye** del tuo Raspberry, compressa in uno zip (vedi la cella 3).

## 1. Prepara l'ambiente

In [ ]:
!git clone -q https://github.com/BotuPotto/KlippEye
%cd KlippEye
!pip install -q -r ml/requirements.txt roboflow
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nessuna (sarà lento: Runtime → Cambia tipo di runtime → GPU)")

## 2. Scarica le raccolte pubbliche
La chiave Roboflow: se l'hai salvata nei **Secret** di Colab (l'icona a chiave a sinistra) col nome `ROBOFLOW_API_KEY`, viene presa da lì (consenti l'accesso quando lo chiede); altrimenti incollala quando te la chiede (non viene mostrata né salvata nel notebook). Ci vogliono alcuni minuti; le raccolte non più disponibili vengono saltate.

In [ ]:
import getpass, os
try:
    from google.colab import userdata
    key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    key = ""
os.environ["ROBOFLOW_API_KEY"] = key or getpass.getpass("Chiave Roboflow: ")
!python ml/download.py

## 3. (Consigliato) Le tue stampe

Sul Raspberry, da un terminale SSH:
```
cd ~/printer_data && zip -r ~/klippeye-data.zip klippeye/recordings klippeye/reviews
```
poi scarica `klippeye-data.zip` sul computer (per esempio con WinSCP o `scp`) e caricalo qui eseguendo la cella sotto.
Se non hai ancora stampe giudicate, salta questa cella.

In [ ]:
from google.colab import files
up = files.upload()
for name in up:
    !unzip -q -o "{name}" -d /content/klippeye-data
print("caricato" if up else "niente caricato")

## 4. L'elenco delle foto
Da ogni raccolta al massimo 3000 foto: i pezzi con il difetto, pezzi della stessa foto lontani dal difetto come esempi di stampa che va bene, la foto intera solo se il difetto ne copre buona parte.

In [ ]:
import os
extra = "--klippeye /content/klippeye-data/klippeye" if os.path.isdir("/content/klippeye-data/klippeye") else ""
!python ml/datasets.py --auto ml/data {extra} --out ml/work

## 5. Addestramento
Con la GPU gratuita di Colab ci vuole circa un'ora (15 giri da 4–6 minuti). A ogni giro vedi come va sulle foto tenute da parte.

In [ ]:
!python ml/train.py --manifest ml/work/manifest.csv --out ml/out/model --epochs 15 --workers 2

## 6. La misura sulle tue stampe
Se hai caricato le tue stampe: per ogni soglia, falsi allarmi all'ora sulle stampe andate bene e stampe fallite prese.

In [ ]:
import os
if os.path.isdir("/content/klippeye-data/klippeye"):
    !python ml/evaluate.py --model ml/out/model --data /content/klippeye-data/klippeye --json ml/out/report.json
else:
    print("Nessuna stampa tua caricata: la misura la farai sul Raspberry.")

## 7. Scarica il modello
Scarica lo zip: dentro ci sono `model.onnx`, `model.json` e l'elenco delle raccolte usate con le loro licenze.

In [ ]:
!cp ml/data/CREDITS.md ml/out/model/ 2>/dev/null; cd ml/out && zip -q -r /content/klippeye-model.zip model && echo pronto
from google.colab import files
files.download("/content/klippeye-model.zip")